# Vietnam results

Produces every table and figure in the paper in one top-to-bottom run.

| Section | Output |
|---|---|
| §0 Setup | paths, stats, land mask, q90/q99 thresholds |
| §1 Models | U-Net, CorrDiff, CorrFlow |
| §2 Helpers | inference and data loading |
| §3 **Table 1** | MAE/CRPS on a month-stratified sample of each split |
| §4 Extreme selection | timesteps whose spatial p99 exceeds the training p99 |
| §5 Extreme inference | cache predictions for the extreme subsets and case timesteps |
| §6 **Tables 2–3** | extreme-event MAE, CRPS, FSS (q99, n=5), BS (q99) |
| §7 **Table 6** | solver ablation: CorrDiff Heun, CorrFlow Euler |
| §8 Map utilities | shared colormap and cartopy setup |
| §10 **Power spectra** | azimuthally averaged PSD on a sample of 2025 test timesteps |
| §11 **Reliability, rank histogram** | reliability at q90/q99, verification rank histogram |
| §11b **Skill distributions** | per-timestep box plots and spread-skill |
| §12 **ODE trajectory** | single-step CorrFlow trajectory |
| §13 **Case studies** | case figures, spread comparison, Table case_metrics |
| §14 Summary | numbers for the LaTeX tables |

**Data conventions**
- Zarr `tp`: ERA5-Land log1p(mm/hr) on the 0.1° grid (the target)
- Zarr `tp_coarse`: ERA5 0.25° log1p(mm/hr), bilinearly interpolated to the 0.1° grid (the model input)
- The model sees z-scored `tp_coarse` and outputs a z-scored prediction
- ERA5 maps use the raw `era5_sl_tp_{year}.nc` on its native 0.25° grid, so the resolution difference is visible
- Predictions: `expm1(z * (S_OUT+1e-6) + M_OUT)` gives mm/hr
- Ground truth: `expm1(zarr["tp"])` gives mm/hr

## §0 Setup

In [ ]:
import os, sys, time, warnings, json, glob, re, struct
import numpy as np
import xarray as xr
import pandas as pd
import torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader
from scipy.ndimage import uniform_filter
from pathlib import Path
from functools import partial

MODULUS_ROOT  = "/home/khaiht/oggy_climate/physicsnemo"   # Modulus v0.9.0 checkout with modulus_patch/ applied
CORRDIFF_ROOT = os.path.join(MODULUS_ROOT, "examples/generative/corrdiff")
for p in [CORRDIFF_ROOT]:
    if p not in sys.path: sys.path.insert(0, p)

from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf, open_dict
from modulus.distributed import DistributedManager
from modulus.launch.logging import PythonLogger, RankZeroLoggingWrapper
from modulus import Module
from modulus.models.diffusion import SongUNetPosEmbd
from modulus.utils.corrdiff import NetCDFWriter, regression_step, diffusion_step
from modulus.utils.generative import deterministic_sampler
from helpers.generate_helpers import get_dataset_and_sampler, save_images
import helpers

# Paths
ZARR_PATH  = "/mnt/data/khaiht/data/vietnam_train_tp_only/vietnam_data.zarr"
STATS_PATH = "/mnt/data/khaiht/data/vietnam_train_tp_only/stats.json"
RAW_TP_DIR = "/mnt/data/khaiht/data/vietnam/input_tp"   # era5_sl_tp_{year}.nc
REG_CKPT   = "/mnt/data/khaiht/outputs/vietnam_regression_hope/checkpoints_regression_best/UNet.0.315136.mdlus"
DIFF_CKPT  = "/mnt/data/khaiht/outputs/vietnam_diffusion_hope/checkpoints_diffusion_best/EDMPrecondSR.0.17310208.mdlus"
FLOW_CKPT  = "/mnt/data/khaiht/outputs/vietnam_flow_hope/checkpoints_flow_best/FlowUNet.0.12225024.pt"
OUT_DIR    = "/mnt/data/khaiht/outputs/vietnam_results_1step"  # updated: 1-step CorrFlow results
os.makedirs(OUT_DIR, exist_ok=True)

# Split years
TRAIN_YEARS = list(range(2017, 2024))
VAL_YEARS   = [2024]
TEST_YEARS  = [2025]
K_ENS       = 16

# Stats (loaded once, used everywhere)
with open(STATS_PATH) as f: _st = json.load(f)
# tp_coarse stats (for normalising model input)
M_IN  = float(_st["input"]["tp_coarse"]["mean"])
S_IN  = float(_st["input"]["tp_coarse"]["std"])
# tp stats (for denormalising model output / ground truth)
M_OUT = float(_st["output"]["tp"]["mean"])
S_OUT = float(_st["output"]["tp"]["std"])
print(f"M_IN={M_IN:.4f}  S_IN={S_IN:.4f}  M_OUT={M_OUT:.4f}  S_OUT={S_OUT:.4f}")

# Distributed (single GPU)
os.environ["MODULUS_DISTRIBUTED_INITIALIZATION_METHOD"] = "ENV"
os.environ["RANK"] = "0"; os.environ["WORLD_SIZE"] = "1"
os.environ["LOCAL_RANK"] = "0"
os.environ["MASTER_ADDR"] = "127.0.0.1"; os.environ["MASTER_PORT"] = "29500"
DistributedManager.initialize()
dist    = DistributedManager()
device  = dist.device
logger0 = RankZeroLoggingWrapper(PythonLogger("results"), dist)
print(f"Device: {device}  |  OUT_DIR: {OUT_DIR}")


In [ ]:
# Open zarr, extract coordinates and land mask
ds_zarr   = xr.open_dataset(ZARR_PATH, engine="zarr", chunks=None)
lat1d_fine = ds_zarr["latitude"].values   # (H,)  fine 0.1° grid
lon1d_fine = ds_zarr["longitude"].values  # (W,)
H, W       = len(lat1d_fine), len(lon1d_fine)
lon2d_fine, lat2d_fine = np.meshgrid(lon1d_fine, lat1d_fine)  # (H,W) for plotting

# Land mask: True where ERA5-Land has valid data (not NaN)
_tp_s    = ds_zarr["tp"].isel(time=0).values
land_mask = (~np.isnan(_tp_s)).astype(bool)  # (H,W)
del _tp_s
print(f"Fine grid: H={H}  W={W}  land={land_mask.sum():,}")

# Training-distribution thresholds (paper: FSS and BS at q90 and q99)
print("Computing training thresholds from zarr ...")
_yr_z  = ds_zarr["time"].dt.year.values
_tr_idx = np.where(np.isin(_yr_z, TRAIN_YEARS))[0]
# Ground truth (target) = zarr "tp" = log1p(mm/hr) of ERA5-Land
_tp_tr  = np.expm1(ds_zarr["tp"].isel(time=_tr_idx).values)  # (T,H,W) mm/hr
_flat   = _tp_tr[~np.isnan(_tp_tr)]
Q90_MM  = float(np.nanquantile(_flat, 0.90))
Q99_MM  = float(np.nanquantile(_flat, 0.99))
del _tp_tr, _flat
print(f"  q90 = {Q90_MM:.4f} mm/hr  |  q99 = {Q99_MM:.4f} mm/hr")


## §1 Load models

In [ ]:
# Models are only needed to fill missing cache entries (cell 14); all figures
# and tables load from precomputed files. Skip this cell if the cache is complete.

# Hydra config
with initialize_config_dir(version_base="1.2",
                           config_dir=f"{CORRDIFF_ROOT}/conf"):
    cfg = compose(config_name="vietnam_config_generate")
OmegaConf.resolve(cfg)
with open_dict(cfg):
    cfg.generation.io.reg_ckpt_filename = REG_CKPT
    cfg.generation.io.res_ckpt_filename = DIFF_CKPT
    cfg.generation.num_ensembles        = K_ENS
    cfg.generation.inference_mode       = "all"

# Instantiate dataset (no specific timesteps yet; used for img_shape / land_mask)
_ds_cfg = OmegaConf.to_container(cfg.dataset)
_ds_cfg["time_range"] = None
dataset, _ = get_dataset_and_sampler(dataset_cfg=_ds_cfg, times=[], has_lead_time=False)
IMG_SHAPE   = dataset.image_shape()   # (H, W)
C_OUT       = len(dataset.output_channels())
C_COND      = len(dataset.input_channels())
assert IMG_SHAPE == (H, W), f"Grid mismatch: zarr ({H},{W}) vs dataset {IMG_SHAPE}"
print(f"IMG_SHAPE={IMG_SHAPE}  C_OUT={C_OUT}  C_COND={C_COND}")

# Models
print("Loading U-Net regression net ...")
net_reg  = Module.from_checkpoint(REG_CKPT).eval().to(device) \
                 .to(memory_format=torch.channels_last)

print("Loading CorrDiff diffusion net ...")
net_diff = Module.from_checkpoint(DIFF_CKPT).eval().to(device) \
                 .to(memory_format=torch.channels_last)

# CorrFlow: must match train_corrflow.py exactly
N_GRID_CH = 4
class FlowUNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = SongUNetPosEmbd(
            img_resolution=IMG_SHAPE,
            in_channels=C_OUT + 1 + C_COND + N_GRID_CH,  # x + t_emb + cond + grid
            out_channels=C_OUT,
            N_grid_channels=N_GRID_CH,
            embedding_type="zero",
        )
    def forward(self, x, t, cond):
        B, _, Hh, Ww = x.shape
        t_emb = t.view(B,1,1,1).expand(B,1,Hh,Ww)
        return self.net(torch.cat([x, t_emb, cond], dim=1),
                        noise_labels=torch.zeros(B, device=x.device),
                        class_labels=None)

print("Loading CorrFlow net ...")
net_flow = FlowUNet().to(device)
_st2 = torch.load(FLOW_CKPT, map_location=device)
if any(k.startswith("module.") for k in _st2):
    _st2 = {k.replace("module.","",1):v for k,v in _st2.items()}
net_flow.load_state_dict(_st2); net_flow = net_flow.eval()
print("All models loaded.")


## §2 Inference helpers

In [ ]:
# Input preparation
# The dataloader (VietnamDataset.__getitem__) does:
#   1. x = zarr["tp_coarse"]  (log1p mm/hr, NaN on ocean)
#   2. x = nan_to_num(x, nan=0.0)
#   3. x = (x - M_IN) / (S_IN + 1e-6)
# The same steps are applied here.

def make_input_tensor(tp_coarse_log1p_hw):
    """
    Prepare model input tensor from zarr tp_coarse (log1p mm/hr, shape H×W).
    Replicates VietnamDataset.__getitem__ exactly.
    Returns: (1, C_COND, H, W) float32 tensor on device, z-score normalised.
    """
    x = np.nan_to_num(tp_coarse_log1p_hw, nan=0.0)   # ocean -> 0 (matches training)
    x = (x - M_IN) / (S_IN + 1e-6)                   # z-score
    return torch.from_numpy(x[np.newaxis, np.newaxis].astype(np.float32)).to(device)


# Output conversion
# Model outputs z-score of log1p(mm/hr).
# Conversion: z -> log1p -> mm/hr = expm1(z * (S_OUT+1e-6) + M_OUT)

def pred_to_mm(pred_z_nhw):
    """
    (n, H, W) z-score → (n, H, W) mm/hr, ocean→NaN.
    pred_z_nhw: numpy array of model output z-scores.
    """
    log1p = pred_z_nhw * (S_OUT + 1e-6) + M_OUT
    mm    = np.clip(np.expm1(log1p), 0.0, None)
    return np.where(land_mask[np.newaxis], mm, np.nan)

def truth_to_mm(tp_log1p_hw):
    """zarr tp (log1p mm/hr, H×W) → mm/hr, ocean→NaN."""
    return np.where(land_mask, np.expm1(tp_log1p_hw), np.nan)


# Generate functions
@torch.no_grad()
def run_unet(img_lr_t):
    """img_lr_t: (1,C_COND,H,W) z-score.  Returns mu (1,C_OUT,H,W) z-score."""
    dummy  = torch.zeros(1, C_OUT, *IMG_SHAPE, device=device)
    sigma0 = torch.zeros(1, device=device)
    return net_reg(dummy, img_lr_t, sigma0)

@torch.no_grad()
def run_corrdiff(img_lr_t, n_steps=50, solver="heun"):
    """Returns full_pred (K,C_OUT,H,W) and mu (1,C_OUT,H,W), both z-score."""
    seeds  = list(range(K_ENS))
    sfn    = partial(deterministic_sampler, num_steps=n_steps, solver=solver)
    mu     = run_unet(img_lr_t)
    cond   = img_lr_t
    if cfg.generation.hr_mean_conditioning:
        cond = torch.cat([mu.expand(img_lr_t.shape[0],-1,-1,-1), img_lr_t], dim=1)
    res = diffusion_step(
        net=net_diff, sampler_fn=sfn,
        seed_batch_size=K_ENS, img_shape=IMG_SHAPE, img_out_channels=C_OUT,
        rank_batches=[torch.tensor(seeds)],
        img_lr=cond.expand(K_ENS,-1,-1,-1).to(memory_format=torch.channels_last),
        rank=0, device=device, hr_mean=None,
    )
    return mu.expand(K_ENS,-1,-1,-1) + res, mu

@torch.no_grad()
def run_corrflow(img_lr_t, n_steps=1, solver="euler",
                 return_intermediates=False, capture_at=None):
    """Returns full_pred (K,C_OUT,H,W) z-score. Optionally also inter dict."""
    mu   = run_unet(img_lr_t)
    cond = img_lr_t.expand(K_ENS, -1, -1, -1)
    r    = torch.randn(K_ENS, C_OUT, *IMG_SHAPE, device=device)
    dt   = 1.0 / n_steps; en_amp = device.type == "cuda"
    inter = {}
    if return_intermediates:
        if capture_at is None: capture_at = list(range(n_steps+1))
        if 0 in capture_at: inter[0] = r.cpu().clone()
    for i in range(n_steps):
        t0 = torch.full((K_ENS,), i/n_steps,     device=device)
        t1 = torch.full((K_ENS,), (i+1)/n_steps, device=device)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=en_amp):
            v0 = net_flow(r, t0, cond)
        if solver == "euler":
            r = r + v0.float() * dt
        else:
            r_p = r + v0.float() * dt
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=en_amp):
                v1 = net_flow(r_p, t1, cond)
            r = r + 0.5*(v0.float()+v1.float()) * dt
        if return_intermediates and (i+1) in capture_at:
            inter[i+1] = r.cpu().clone()
    pred = mu.expand(K_ENS,-1,-1,-1) + r
    return (pred, mu, inter) if return_intermediates else (pred, mu)


In [ ]:
# ERA5 raw 0.25° loader for visualisation
# Returns ERA5 tp on its native 0.25° coarse grid (not regridded) so that
# pcolormesh displays the blocky 0.25° resolution in map figures.

def load_era5_coarse_native(ts_str):
    """
    Load ERA5 0.25° raw TP at ts_str from era5_sl_tp_{year}.nc.
    Returns:
      era5_mm      : (H_c, W_c) mm/hr on the native 0.25° grid (NaN = ocean via land mask)
      era5_lat1d   : (H_c,) latitude array for the coarse grid
      era5_lon1d   : (W_c,) longitude array for the coarse grid
    The caller passes era5_lat1d / era5_lon1d to pcolormesh so the blocky
    0.25° structure is visible rather than smooth-interpolated.
    """
    ts  = pd.Timestamp(ts_str)
    fp  = f"{RAW_TP_DIR}/era5_sl_tp_{ts.year}.nc"
    ds  = xr.open_dataset(fp)
    if "valid_time" in ds.dims: ds = ds.rename({"valid_time": "time"})
    ds_t = ds.sel(time=ts, method="nearest")
    tp_c  = np.clip(ds_t["tp"].values * 1000.0, 0.0, None)  # m -> mm/hr
    c_lat = ds_t["latitude"].values.copy()
    c_lon = ds_t["longitude"].values.copy()
    ds.close()
    # Clip to Vietnam domain with a small margin
    lat_mask = (c_lat >= 5.0) & (c_lat <= 26.0)
    lon_mask = (c_lon >= 100.0) & (c_lon <= 120.0)
    tp_c  = tp_c[np.ix_(lat_mask, lon_mask)]
    c_lat = c_lat[lat_mask]
    c_lon = c_lon[lon_mask]
    return tp_c.astype(np.float32), c_lat, c_lon


# Cache helpers
# Each timestep cached as .npz with:
#   truth_mm  (H,W)    : ERA5-Land mm/hr, ocean=NaN
#   unet_mm   (H,W)    : U-Net prediction mm/hr, ocean=NaN
#   cd_mm     (K,H,W)  : CorrDiff ensemble mm/hr, ocean=NaN
#   cf_mm     (K,H,W)  : CorrFlow ensemble mm/hr, ocean=NaN
# ERA5 raw not cached (cheap to reload, avoids storing coarse+fine pair)

def _cache_path(split, zi):
    d = Path(OUT_DIR) / "cache" / split
    d.mkdir(parents=True, exist_ok=True)
    return str(d / f"{zi}.npz")

def _load_cache(split, zi):
    p = _cache_path(split, zi)
    if not os.path.exists(p): return None
    d = np.load(p)
    return d["truth_mm"], d["unet_mm"], d["cd_mm"], d["cf_mm"]

def _run_and_cache(split, zi):
    """Run all models at zarr index zi and save to cache. Returns cached tuple."""
    tp_target_log1p  = ds_zarr["tp"].isel(time=int(zi)).values         # ERA5-Land log1p
    tp_coarse_log1p  = ds_zarr["tp_coarse"].isel(time=int(zi)).values  # ERA5 coarse log1p

    truth_mm = truth_to_mm(tp_target_log1p)                             # (H,W)
    img_lr_t = make_input_tensor(tp_coarse_log1p)                       # (1,1,H,W) z-score

    mu_z                = run_unet(img_lr_t)
    unet_mm             = pred_to_mm(mu_z.cpu().numpy()[:,0,:,:])       # (1,H,W) -> NaN ocean
    unet_mm             = unet_mm[0]                                    # (H,W)

    cd_z, _             = run_corrdiff(img_lr_t)
    cd_mm               = pred_to_mm(cd_z.cpu().numpy()[:,0,:,:])       # (K,H,W)

    cf_z, _             = run_corrflow(img_lr_t)
    cf_mm               = pred_to_mm(cf_z.cpu().numpy()[:,0,:,:])       # (K,H,W)

    np.savez_compressed(_cache_path(split, zi),
                        truth_mm=truth_mm.astype(np.float32),
                        unet_mm=unet_mm.astype(np.float32),
                        cd_mm=cd_mm.astype(np.float32),
                        cf_mm=cf_mm.astype(np.float32))
    return truth_mm, unet_mm, cd_mm, cf_mm

def _get(split, zi):
    """Load from cache or compute."""
    c = _load_cache(split, zi)
    return c if c is not None else _run_and_cache(split, zi)

print("Helpers defined.")


## §3 Table 1: broad-sample MAE and CRPS
Computed from the month-stratified sample cache written by
`evaluation/cache_sample.py`; no inference or GPU needed.


In [ ]:
# Table 1: full-period metrics from the month-stratified sample cache
# cache_sample.py caches the ensembles in cache/{split}/{zi}.npz and
# writes fullperiod_sample_idxs.json; metrics are computed here without a GPU.
# CRPS uses the corrected estimator, without the spurious factor of 0.5.
FULL_CSV    = os.path.join(OUT_DIR, "tab1_full_period.csv")
SAMPLE_JSON = os.path.join(OUT_DIR, "fullperiod_sample_idxs.json")

def _crps_fixed(ens_mm, truth_mm):
    """(K,H,W) ensemble, (H,W) truth -> scalar CRPS over valid land pixels."""
    valid = ~np.isnan(truth_mm) & ~np.all(np.isnan(ens_mm), axis=0)
    if valid.sum() == 0:
        return np.nan
    k   = ens_mm.shape[0]
    obs = truth_mm[valid]
    ens = ens_mm[:, valid]
    if k == 1:
        return float(np.mean(np.abs(ens[0] - obs)))
    es  = np.sort(ens, axis=0)
    idx = np.arange(k).reshape(k, 1)
    sp  = np.sum((2 * idx - k + 1) * es, axis=0)
    acc = np.mean(np.abs(ens - obs[np.newaxis]), axis=0)
    return float(np.mean(acc - sp / k**2))           # corrected estimator: dispersion term not halved

def _cache_load(split, zi):
    p = os.path.join(OUT_DIR, "cache", split, f"{int(zi)}.npz")
    if not os.path.exists(p):
        return None
    d = np.load(p)
    return d["truth_mm"], d["unet_mm"], d["cd_mm"], d["cf_mm"]

assert os.path.exists(SAMPLE_JSON), (
    f"{SAMPLE_JSON} not found — run evaluation/cache_sample.py first.")
with open(SAMPLE_JSON) as fh:
    _fp = json.load(fh)
fp_idxs = {k: np.array(v) for k, v in _fp["sample_idxs"].items()}
print(f"Sample (seed={_fp.get('seed')}): "
      + ", ".join(f"{k}={len(v)}" for k, v in fp_idxs.items()))

rows = []
for split_name in ("train", "val", "test"):
    idxs = fp_idxs.get(split_name, [])
    acc  = {k: {"mae": 0., "crps": 0., "n": 0} for k in ("bl", "un", "cd", "cf")}
    miss = 0
    for zi in idxs:
        loaded = _cache_load(split_name, zi)
        if loaded is None:
            miss += 1
            continue
        truth, unet, cd, cf = loaded
        tp_c = ds_zarr["tp_coarse"].isel(time=int(zi)).values
        era5 = np.where(land_mask, np.expm1(tp_c), np.nan)
        for tag, pm, ens in [
            ("bl", era5,              era5[np.newaxis]),
            ("un", unet,              unet[np.newaxis]),
            ("cd", np.nanmean(cd, 0), cd),
            ("cf", np.nanmean(cf, 0), cf),
        ]:
            mae  = float(np.nanmean(np.abs(pm - truth)))
            crps = _crps_fixed(ens, truth)
            if np.isnan(mae) or np.isnan(crps):
                continue
            acc[tag]["mae"]  += mae
            acc[tag]["crps"] += crps
            acc[tag]["n"]    += 1
    if miss:
        print(f"  {split_name}: {miss} sampled timesteps not in cache (skipped)")
    for key, label in [("bl", "ERA5-Bilinear"), ("un", "CorrDiff-UNet"),
                       ("cd", "CorrDiff (Heun)"), ("cf", "CorrFlow (Euler)")]:
        n_v = acc[key]["n"]
        rows.append({"split": split_name, "model": label,
                     "MAE":  acc[key]["mae"]  / max(n_v, 1),
                     "CRPS": acc[key]["crps"] / max(n_v, 1),
                     "n_timesteps": n_v})

df_full = pd.DataFrame(rows)
df_full.to_csv(FULL_CSV, index=False)
print("\n=== Table 1: Full-period (month-stratified sample) MAE / CRPS ===")
print(df_full.round(4).to_string(index=False))


## §4 Extreme timestep selection

In [ ]:
# Load pre-computed metadata from cache_extremes.py if available,
# otherwise compute from scratch (requires GPU + ~5 min zarr scan).
_meta_path = os.path.join(OUT_DIR, "extreme_meta.json")
if os.path.exists(_meta_path):
    with open(_meta_path) as fh:
        _meta = json.load(fh)
    Q90_MM       = float(_meta["Q90_MM"])
    Q99_MM       = float(_meta["Q99_MM"])
    _yrs_zarr    = ds_zarr["time"].dt.year.values
    _times_zarr  = ds_zarr["time"].values
    extreme_idxs = {k: np.array(v) for k, v in _meta["extreme_idxs"].items()}
    print(f"Loaded from extreme_meta.json  (q99={Q99_MM:.4f} mm/hr)")
    for k, v in extreme_idxs.items():
        print(f"  {k}: {len(v):,} extreme timesteps")
    CASE_META = {
        # Three held-out 2025 test events, chronological (seasonal progression).
        "case1": ("2025-04-28T02:00:00", "test", (19.0,24.0,102.1,107.1), (21.5,104.6), "fig_case_2025apr"),   # N pre-monsoon
        "case2": ("2025-08-25T18:00:00", "test", (15.9,20.9,102.4,107.4), (18.4,104.9), "fig_case_2025aug"),   # N-central SW monsoon
        "case3": ("2025-11-06T13:00:00", "test", (11.2,16.2,106.6,111.6), (13.7,109.1), "fig_case_2025nov"),   # S-central NE monsoon
    }
else:
    # Compute from scratch
    # Paper: timesteps where spatial 99th percentile of ERA5-Land tp > training p99 threshold
    print(f"Selecting extremes using Q99_MM = {Q99_MM:.4f} mm/hr ...")
    _yrs_zarr   = ds_zarr["time"].dt.year.values
    _times_zarr = ds_zarr["time"].values
    extreme_idxs = {}
    
    for split_name, split_years in [("train",TRAIN_YEARS),("val",VAL_YEARS),("test",TEST_YEARS)]:
        idxs_s    = np.where(np.isin(_yrs_zarr, split_years))[0]
        # spatial p99 of ERA5-Land tp (the target)
        tp_log_s  = ds_zarr["tp"].isel(time=idxs_s).values            # (T,H,W) log1p
        spat_q    = np.nanquantile(np.expm1(tp_log_s), 0.99, axis=(1,2))  # (T,)
        sel       = np.where(spat_q > Q99_MM)[0]
        extreme_idxs[split_name] = idxs_s[sel]
        print(f"  {split_name}: {len(sel):,} extreme timesteps (of {len(idxs_s):,})")
    
    CASE_META = {
        # Three held-out 2025 test events, chronological (seasonal progression).
        "case1": ("2025-04-28T02:00:00", "test", (19.0,24.0,102.1,107.1), (21.5,104.6), "fig_case_2025apr"),   # N pre-monsoon
        "case2": ("2025-08-25T18:00:00", "test", (15.9,20.9,102.4,107.4), (18.4,104.9), "fig_case_2025aug"),   # N-central SW monsoon
        "case3": ("2025-11-06T13:00:00", "test", (11.2,16.2,106.6,111.6), (13.7,109.1), "fig_case_2025nov"),   # S-central NE monsoon
    }
    

## §5 Extreme-subset inference (cached to disk)

In [ ]:
# Cache files written by cache_extremes.py are loaded automatically by _get().
# This cell only runs inference for any timestep not yet cached
# (e.g. if the Slurm job was interrupted partway through).
_times_pd = pd.to_datetime(_times_zarr)

for run_split, run_idxs in [
    ("test",  extreme_idxs["test"]),
    ("train", extreme_idxs["train"]),
]:
    n_missing = sum(
        1 for zi in run_idxs
        if not os.path.exists(_cache_path(run_split, zi))
    )
    if n_missing == 0:
        print(f"{run_split} extreme: all {len(run_idxs):,} timesteps cached. ✅")
        continue
    print(f"{run_split} extreme: {n_missing}/{len(run_idxs)} timesteps not cached — running ...")
    t0 = time.time()
    for ii, zi in enumerate(run_idxs):
        if _load_cache(run_split, zi) is not None:
            continue
        _run_and_cache(run_split, zi)
        if (ii + 1) % 10 == 0:
            rate = (time.time() - t0) / (ii + 1)
            print(f"  {ii+1}/{len(run_idxs)}  {rate:.1f} s/ts", flush=True)
    print(f"  Done in {(time.time()-t0)/60:.1f} min.")

for cname, (cts, split_cs, _zoom, _peak, _fname) in CASE_META.items():
    if cts is None:
        continue
    zi_c = int(np.argmin(np.abs(_times_pd - pd.Timestamp(cts))))
    if _load_cache(split_cs, zi_c) is None:
        print(f"Running {cname} ({cts}) ...")
        _run_and_cache(split_cs, zi_c)
    else:
        print(f"{cname}: cached ✅")

print("\n✅ All inference ready.")


## §6 Tables 2 and 3: extreme-event metrics

In [ ]:
def _crps_ens(truth_mm_hw, ens_mm_khw):
    """(H,W) truth, (K,H,W) ens → scalar CRPS over land."""
    k = ens_mm_khw.shape[0]
    if k == 1: return float(np.nanmean(np.abs(ens_mm_khw[0]-truth_mm_hw)))
    es  = np.sort(ens_mm_khw, axis=0)
    idx = np.arange(k).reshape(k,1,1)
    sp  = np.nansum((2*idx-k+1)*es, axis=0)
    acc = np.nanmean(np.abs(ens_mm_khw - truth_mm_hw[np.newaxis]), axis=0)
    return float(np.nanmean(acc - sp/k**2))

def _fss(truth_hw, pred_mean_hw, thr, scale=5):
    ob = np.nan_to_num(np.where(np.isnan(truth_hw),    np.nan, (truth_hw    >=thr).astype(float)), nan=0.)
    pr = np.nan_to_num(np.where(np.isnan(pred_mean_hw),np.nan, (pred_mean_hw>=thr).astype(float)), nan=0.)
    sz = 2*scale+1
    of = uniform_filter(ob, size=sz); pf = uniform_filter(pr, size=sz)
    lv = ~np.isnan(truth_hw)
    mse = np.mean((of-pf)[lv]**2)
    ref = np.mean(of[lv]**2) + np.mean(pf[lv]**2)
    return 1.0 if ref<1e-12 else float(1.0-mse/ref)

def _bs(truth_hw, ens_mm_khw, thr):
    with warnings.catch_warnings(): warnings.simplefilter("ignore")
    exc  = np.where(land_mask, np.nanmean((ens_mm_khw>=thr).astype(float),axis=0), np.nan)
    obin = np.where(np.isnan(truth_hw), np.nan, (truth_hw>=thr).astype(float))
    return float(np.nanmean((exc-obin)**2))

def compute_extreme_metrics(split, idxs):
    acc = {k: {"mae":0.,"crps":0.,"fss":0.,"bs":0.} for k in ("bl","un","cd","cf")}
    n = 0
    for zi in idxs:
        truth, unet, cd, cf = _get(split, zi)
        # ERA5-Bilinear: expm1(zarr tp_coarse) on fine grid
        tp_c_log = ds_zarr["tp_coarse"].isel(time=int(zi)).values
        era5_fine = np.where(land_mask, np.expm1(tp_c_log), np.nan)
        m_cd = np.nanmean(cd, axis=0); m_cf = np.nanmean(cf, axis=0)
        for tag, pm, ens in [
            ("bl", era5_fine, era5_fine[np.newaxis]),
            ("un", unet,      unet[np.newaxis]),
            ("cd", m_cd,      cd),
            ("cf", m_cf,      cf),
        ]:
            acc[tag]["mae"]  += float(np.nanmean(np.abs(pm - truth)))
            acc[tag]["crps"] += _crps_ens(truth, ens)
            acc[tag]["fss"]  += _fss(truth, pm, Q99_MM, scale=5)
            acc[tag]["bs"]   += _bs(truth, ens, Q99_MM)
        n += 1
    return {k: {m: v/n for m,v in d.items()} for k,d in acc.items()}

In [ ]:
print("Computing extreme metrics ...")
m_test  = compute_extreme_metrics("test",  extreme_idxs["test"])
m_train = compute_extreme_metrics("train", extreme_idxs["train"])

label_map = {"bl":"ERA5-Bilinear","un":"CorrDiff-UNet","cd":"CorrDiff (Heun)","cf":"CorrFlow (Euler)"}
def _show(m, title):
    print(f"\n=== {title} ===")
    for k in ("bl","un","cd","cf"):
        d = m[k]
        print(f"  {label_map[k]:<22}  MAE={d['mae']:.3f}  CRPS={d['crps']:.3f}"
              f"  FSS={d['fss']:.3f}  BS={d['bs']:.3f}")

_show(m_test,  "Table 2: Extreme test (2025)")
_show(m_train, "Table 3: Extreme train (2017-2023)")

pd.DataFrame(m_test ).T.to_csv(os.path.join(OUT_DIR,"tab2_extreme_test.csv"))
pd.DataFrame(m_train).T.to_csv(os.path.join(OUT_DIR,"tab3_extreme_train.csv"))


## §7 Table 6: solver ablation

In [ ]:
# Table 6: solver ablation (CorrDiff Heun, CorrFlow Euler)
# Written by solver_ablation.py.
_ab_csv = os.path.join(OUT_DIR, "tab6_ablation.csv")
assert os.path.exists(_ab_csv), (
    f"tab6_ablation.csv not found.\n"
    f"Run: python -u evaluation/solver_ablation.py\n{_ab_csv}")

df_ab = pd.read_csv(_ab_csv)
print("=== Table 6: Solver ablation ===")
print(df_ab[["model", "solver", "steps", "MAE", "CRPS", "Latency_s"]]
      .round(4).to_string(index=False))


## §8 Map utilities (shared by all figures)

In [ ]:
_LAND  = cfeature.NaturalEarthFeature("physical","land","10m",
             facecolor="#f5f5f0",edgecolor="none",zorder=0)
_COAST = cfeature.NaturalEarthFeature("physical","coastline","10m",
             facecolor="none",edgecolor="#222222",linewidth=0.8,zorder=1)
_BORDS = cfeature.NaturalEarthFeature("cultural","admin_0_boundary_lines_land","10m",
             facecolor="none",edgecolor="#555555",linewidth=0.6,zorder=2)
_ISLES = [(16.50,112.00,"Hoàng Sa","(Paracel Is.)"),(9.90,114.20,"Trường Sa","(Spratly Is.)")]
_city_reader = None
def _cities():
    global _city_reader
    if _city_reader is None:
        _city_reader = shapereader.Reader(
            shapereader.natural_earth("10m","cultural","populated_places"))
    return _city_reader

RAIN_BOUNDS = [0.1,0.5,1.,2.,5.,10.,20.,40.,80.,150.]
RAIN_COLORS = ["#c8eeff","#75c6f5","#2196c4","#65d47e",
               "#f5e642","#f5a623","#e84c2b","#b01a1a","#6b0f0f"]
RAIN_CMAP   = mcolors.ListedColormap(RAIN_COLORS)
RAIN_CMAP.set_under("#f7f7f7"); RAIN_CMAP.set_over("#3d0000")
RAIN_NORM   = mcolors.BoundaryNorm(RAIN_BOUNDS, ncolors=RAIN_CMAP.N)
ERR_CMAP    = plt.cm.RdBu_r.copy(); ERR_CMAP.set_bad("#d9d9d9")

def _setup_ax(ax, proj, extent, gridlines=True):
    ax.set_facecolor(OCEAN_COLOR)
    ax.add_feature(_LAND); ax.add_feature(_COAST); ax.add_feature(_BORDS)
    ax.set_extent(extent, crs=proj)
    if gridlines:
        gl = ax.gridlines(draw_labels=True,linewidth=0.3,alpha=0.4,
                          linestyle="--",color="gray")
        gl.top_labels=False; gl.right_labels=False
        gl.xlabel_style={"size":12}; gl.ylabel_style={"size":12}
        step = 1.0 if (extent[1]-extent[0]) < 8 else 2.0
        gl.xlocator = mticker.FixedLocator(np.arange(100,120,step))
        gl.ylocator = mticker.FixedLocator(np.arange(5,27,step))
    show_es = extent[0]<113 and extent[3]>13
    for la, lo, vn, en in _ISLES:
        if extent[0] < lo < extent[1] and extent[2] < la < extent[3]:
            if vn == 'Trường Sa':
                # Truong Sa sits near the panel's bottom-right corner, so its label keeps a
                # fixed offset.
                dot_la, dot_lo = la + 0.6, lo + 0.6
                ax.text(dot_lo + 0.25, dot_la + 0.25, f'{vn}\n{en}', transform=proj,
                        fontsize=8, color='#111', zorder=9,
                        bbox=dict(fc='white', alpha=0.7, ec='#aaa', lw=0.3,
                                pad=1, boxstyle='round,pad=0.3'))
            else:
                ax.text(lo + 0.25, la + 0.25, f'{vn}\n{en}', transform=proj, fontsize=8,
                        color='#111', zorder=9,
                        bbox=dict(fc='white', alpha=0.7, ec='#aaa', lw=0.3,
                                pad=1, boxstyle='round,pad=0.3'))
    for rec in _cities().records():
        lo_c,la_c = rec.geometry.x, rec.geometry.y
        pop = rec.attributes.get("POP_MAX",0) or 0
        if extent[0]<lo_c<extent[1] and extent[2]<la_c<extent[3] and pop>200_000:
            ax.plot(lo_c,la_c,"o",ms=3.5,color="red",mec="white",mew=0.4,
                    transform=proj,zorder=7)
            ax.text(lo_c+0.1,la_c+0.1,rec.attributes.get("NAME",""),fontsize=7,
                    transform=proj,zorder=8,
                    bbox=dict(fc="white",alpha=0.55,ec="none",pad=0.8))
    return gl if gridlines else None

FULL_EXTENT = [102.,118.,7.,24.5]
OCEAN_COLOR = "#d0e8f5"  # soft ocean blue for NaN/ocean pixels
print("Map utilities ready.")

In [ ]:
# Study-area figure (Dataset section)
# Defines its own _setup_study_ax so it doesn't affect the shared _setup_ax
# used by the case-study and diagnostic figures.
#
# Terrain is ERA5 surface geopotential (elevation = z / g, 0.25 deg), which
# shows the Annamite Range and the two deltas.
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature

plt.rcParams["pdf.fonttype"] = 42          # embed TrueType, keep text vector

DEM_NC   = os.path.join(OUT_DIR, "era5_orography.nc")
DEM_AREA = [26.5, 100.0, 4.5, 120.0]       # N, W, S, E  (wider than the model domain)

def _load_elevation():
    """Elevation (m) from ERA5 surface geopotential: elev = z / g.

    NOTE: ERA5-Land does NOT expose geopotential through the CDS API (its
    orography is only published as a static attachment in the documentation).
    ERA5 single-levels does, at 0.25 deg, which is ample for a study-area map.
    Returns (elev, lats, lons) or (None, None, None).
    """
    if not os.path.exists(DEM_NC):
        try:
            import cdsapi
            print("  downloading ERA5 surface geopotential (one-off) ...")
            req = {"product_type": ["reanalysis"],
                   "variable": ["geopotential"],
                   "year": ["2020"], "month": ["01"], "day": ["01"],
                   "time": ["00:00"],
                   "area": DEM_AREA,
                   "grid": [0.25, 0.25],
                   "data_format": "netcdf",
                   "download_format": "unarchived"}
            try:
                cdsapi.Client().retrieve("reanalysis-era5-single-levels", req, DEM_NC)
            except Exception:
                # older CDS API: legacy keys
                req.pop("download_format", None)
                req["format"] = req.pop("data_format")
                cdsapi.Client().retrieve("reanalysis-era5-single-levels", req, DEM_NC)
        except Exception as e:
            print(f"  DEM unavailable ({e}); falling back to flat land fill")
            return None, None, None
    try:
        d = xr.open_dataset(DEM_NC)
        zname = "z" if "z" in d.data_vars else list(d.data_vars)[0]
        z = d[zname]
        for tdim in ("valid_time", "time", "number", "expver"):
            if tdim in z.dims:
                z = z.isel({tdim: 0})
        elev = (z / 9.80665).values.astype("float32")
        lats = z["latitude"].values
        lons = z["longitude"].values
        d.close()
        print(f"  elevation loaded: {elev.shape}, range {np.nanmin(elev):.0f}"
              f" to {np.nanmax(elev):.0f} m")
        return elev, lats, lons
    except Exception as e:
        print(f"  could not read {DEM_NC} ({e}); flat land fill")
        return None, None, None

# a muted land ramp: green lowlands -> tan uplands -> pale ridge tops
_TERRAIN = mcolors.LinearSegmentedColormap.from_list(
    "vn_terrain", ["#dfeacb", "#c3d8a4", "#cfc48f", "#b39b6e", "#8f6f52", "#6b5340"])

# cities mentioned in the paper's regime discussion
_KEY_CITIES = [(21.03, 105.85, "Hanoi"), (16.46, 107.59, "Hue"),
               (16.05, 108.22, "Da Nang"), (13.77, 109.22, "Qui Nhon"),
               (10.82, 106.63, "Ho Chi Minh City")]
_ISLES_S = [(16.50, 112.00, "Hoàng Sa", "(Paracel Is.)"),
            (9.90, 114.20, "Trường Sa", "(Spratly Is.)")]

def _setup_study_ax(ax, proj, extent):
    """Map furniture for THIS figure only. No automatic city dump."""
    ax.set_extent(extent, crs=proj)
    ax.add_feature(cfeature.NaturalEarthFeature("physical", "ocean", "10m",
                   facecolor="#d0e8f5", edgecolor="none"), zorder=3)
    ax.add_feature(cfeature.NaturalEarthFeature("physical", "lakes", "10m",
                   facecolor="#cfe3f2", edgecolor="none"), zorder=4)
    ax.add_feature(cfeature.NaturalEarthFeature("physical", "rivers_lake_centerlines",
                   "10m", facecolor="none", edgecolor="#8fb8d6", linewidth=0.5), zorder=5)
    ax.add_feature(cfeature.NaturalEarthFeature("physical", "coastline", "10m",
                   facecolor="none", edgecolor="#333333", linewidth=0.7), zorder=6)
    ax.add_feature(cfeature.NaturalEarthFeature("cultural",
                   "admin_0_boundary_lines_land", "10m", facecolor="none",
                   edgecolor="#666666", linewidth=0.6, linestyle=":"), zorder=6)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4,
                      linestyle="--", color="gray")
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = {"size": 11}; gl.ylabel_style = {"size": 11}
    gl.xlocator = mticker.FixedLocator(np.arange(100, 122, 4))
    gl.ylocator = mticker.FixedLocator(np.arange(4, 28, 4))
    # neutral sea + island labelling, consistent with the other maps
    _isle_bbox = dict(facecolor="white", alpha=0.7, edgecolor="#aaaaaa",
                       linewidth=0.3, pad=1, boxstyle="round,pad=0.3")
    for la, lo, vn, en in _ISLES_S:
        if vn == "Trường Sa":
            dot_lo, dot_la = lo + 0.6, la + 0.6
        else:
            dot_lo, dot_la = lo, la
        ax.text(dot_lo + 0.25, dot_la + 0.25, f"{vn}\n{en}", transform=proj, fontsize=7.5,
                color="#111", zorder=9, bbox=_isle_bbox)
    return gl

CASES = [(21.5, 104.6, "Case 1\n28 Apr"), (18.4, 104.9, "Case 2\n25 Aug"),
         (14.2, 108.8, "Case 3\n6 Nov")]
DOM_LAT, DOM_LON = (5.8, 25.0), (102.0, 118.0)
MAP_EXTENT = [100.0, 120.0, 4.5, 26.5]      # wider than the domain, so the box shows

proj = ccrs.PlateCarree()
fig = plt.figure(figsize=(8.6, 9.6))
ax = fig.add_subplot(1, 1, 1, projection=proj)

elev, dlat, dlon = _load_elevation()

# Land underlay: masked (near-sea-level) terrain pixels fall back to a land tint
# instead of showing the white page. Drawn below the terrain mesh.
ax.add_feature(cfeature.NaturalEarthFeature("physical", "land", "10m",
               facecolor="#e4eed3", edgecolor="none"), zorder=0.5)

if elev is not None:
    elev = np.where(np.isfinite(elev), elev, np.nan)
    elev_land = np.where(elev > 1.0, elev, np.nan)   # sea/geoid noise -> transparent
    im = ax.pcolormesh(dlon, dlat, elev_land, transform=proj,
                       cmap=_TERRAIN, vmin=0, vmax=2500, shading="auto",
                       zorder=1, rasterized=True)
else:
    ax.add_feature(cfeature.NaturalEarthFeature("physical", "land", "10m",
                   facecolor="#eeeadf", edgecolor="none"), zorder=1)
    im = None

_setup_study_ax(ax, proj, MAP_EXTENT)

# model domain box, inside the frame
ax.add_patch(mpatches.Rectangle((DOM_LON[0], DOM_LAT[0]),
             DOM_LON[1]-DOM_LON[0], DOM_LAT[1]-DOM_LAT[0], transform=proj,
             facecolor="none", edgecolor="#c62828", linewidth=2.0,
             linestyle="--", zorder=11))
ax.text(DOM_LON[0] + 0.3, DOM_LAT[1] + 0.35, "model domain", transform=proj,
        fontsize=10.5, color="#c62828", fontweight="bold", zorder=12)

# physiography labels
for la, lo, txt, rot in [(20.5, 106.0, "Red River\nDelta", 0),
                         (17.4, 106.0, "Annamite Range", -58),
                         (9.9, 105.5, "Mekong\nDelta", 0)]:
    ax.text(lo, la, txt, transform=proj, fontsize=9, style="italic", color="#222222",
            ha="center", va="center", rotation=rot, zorder=10,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.6))

for la, lo, name in _KEY_CITIES:
    ax.plot(lo, la, "o", ms=3.8, color="#b3202c", mec="white", mew=0.5,
            transform=proj, zorder=10)
    ax.text(lo + 0.18, la + 0.12, name, fontsize=8, transform=proj, zorder=10,
            bbox=dict(fc="white", alpha=0.6, ec="none", pad=0.6))

# per-case label anchoring: cases 1-2 sit inland (label to the upper-left),
# case 3 is coastal (label to the right, over open water) so it clears the
# rotated "Annamite Range" text on the ridge.
_CASE_ANCHOR = {"Case 1": (-0.5, 0.5, "right", "bottom"),
                "Case 2": (-0.5, 0.5, "right", "bottom"),
                "Case 3": (0.6, 0.35, "left", "bottom")}
for la, lo, label in CASES:
    ax.plot(lo, la, marker="*", color="#ffd633", ms=16, mec="black", mew=0.8,
            transform=proj, zorder=13)
    dx, dy, ha, va = _CASE_ANCHOR[label.split("\n")[0]]
    ax.text(lo + dx, la + dy, label, transform=proj, fontsize=9, fontweight="bold",
            ha=ha, va=va, zorder=13,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#999", lw=0.4, alpha=0.9))

if im is not None:
    cb = fig.colorbar(im, ax=ax, shrink=0.55, pad=0.03, extend="max", location="right")
    cb.set_label("elevation (m)", fontsize=11); cb.ax.tick_params(labelsize=9)

out_f = os.path.join(OUT_DIR, "study_area.pdf")
fig.savefig(out_f, dpi=300, bbox_inches="tight")
fig.savefig(out_f.replace(".pdf", ".png"), dpi=300, bbox_inches="tight")
plt.close(fig)
print("  study_area.pdf + .png")

## §10 Figure: power spectra

In [ ]:
# Figure power_spectra
# PSD data written by power_spectra.py.
_psd_path = os.path.join(OUT_DIR, "psd_results.npz")
assert os.path.exists(_psd_path), (
    f"psd_results.npz not found.\n"
    f"Run: python -u evaluation/power_spectra.py\n{_psd_path}")

_psd = np.load(_psd_path)
freq_ref = _psd["freq"]

# Key mapping: spaces and parens replaced with underscores when saved
_KEY_MAP = {
    "ERA5-Land":        "ERA5-Land",
    "ERA5-Bilinear":    "ERA5-Bilinear",
    "U-Net":             "CorrDiff-UNet",
    "CorrDiff":  "CorrDiff_Heun",
    "CorrFlow": "CorrFlow_Euler",
}
psd_mean = {label: (freq_ref, _psd[key]) for label, key in _KEY_MAP.items()}

STYLES = {
    "ERA5-Land":        ("#FFD500", "-",  6.5),   # bright yellow target (drawn at back)
    "ERA5-Bilinear":    ("#888888", "--", 1.8),
    "U-Net":             ("#7B1FA2", "-",  1.8),            # U-Net: purple
    "CorrDiff":  ("#FF9800", "-",  1.8),   # CorrDiff: orange
    "CorrFlow": ("#E53935", "-",  2.0),   # CorrFlow: red
}
n_all = int(_psd.get("n_timesteps", 0)) if "n_timesteps" in _psd else "all"

fig, ax = plt.subplots(figsize=(8, 5))
for label, (fr, pr) in psd_mean.items():
    c, ls, lw = STYLES[label]
    m = fr > 0
    ax.loglog(fr[m], pr[m], color=c, ls=ls, lw=lw, label=label,
              zorder=(1 if label == "ERA5-Land" else 3))
ax.set_xlabel("Spatial frequency (cycles/km)", fontsize=11)
ax.set_ylabel("Power spectral density",        fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
psd_path = os.path.join(OUT_DIR, "power_spectra.pdf")
fig.savefig(psd_path, dpi=300, bbox_inches="tight")
fig.savefig(psd_path.replace(".pdf", ".png"), dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"✅ {psd_path}")


## §11 Figures: reliability diagrams and rank histogram

In [ ]:
# Distributional diagnostics: reliability and rank histogram (separate figures)
# Reliability at the 90th/99th-percentile thresholds and a verification-rank histogram
# over wet land pixels, both on the 2025 extreme subset. The power spectrum is produced
# separately in the PSD cell above. Saves reliability.pdf and rank_histogram.pdf.
colors = {"CorrDiff": "#FF9800", "CorrFlow": "#E53935"}

def _reliability_curve(idxs, split, model_key, thr, n_bins=10):
    prob_all, obs_all = [], []
    for zi in idxs:
        truth, unet, cd, cf = _get(split, zi)
        ens   = cd if model_key == "cd" else cf
        obs_l = np.where(land_mask, truth, np.nan)
        ens_l = np.where(land_mask[np.newaxis], ens, np.nan)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ep = np.nanmean((ens_l >= thr).astype(float), axis=0)
        eo = (obs_l >= thr).astype(float)
        v  = ~np.isnan(obs_l)
        prob_all.append(ep[v].ravel()); obs_all.append(eo[v].ravel())
    prob = np.concatenate(prob_all); obs = np.concatenate(obs_all)
    bins = np.linspace(0, 1, n_bins + 1); ctr = 0.5 * (bins[:-1] + bins[1:])
    freq = np.full(n_bins, np.nan); cnt = np.zeros(n_bins, int)
    for i in range(n_bins):
        m = ((prob >= bins[i]) & (prob < bins[i + 1])) if i < n_bins - 1 \
            else ((prob >= bins[i]) & (prob <= bins[i + 1]))
        cnt[i] = int(m.sum())
        if cnt[i] > 0:
            freq[i] = obs[m].mean()
    return ctr, freq, cnt

def _rank_hist(idxs, split, model_key, wet_thr=0.1):
    """Verification-rank histogram over WET land pixels (truth > wet_thr mm/hr).
    Conditioning on wet pixels avoids the zero-inflation that makes an all-pixel
    precip rank histogram meaningless; ties are broken randomly."""
    K = K_ENS
    counts = np.zeros(K + 1, dtype=np.int64)
    rng = np.random.default_rng(0)
    for zi in idxs:
        truth, unet, cd, cf = _get(split, zi)
        ens = cd if model_key == "cd" else cf
        v = (~np.isnan(truth)) & (truth > wet_thr) & (~np.any(np.isnan(ens), axis=0))
        if not v.any():
            continue
        o = truth[v]; e = ens[:, v]
        below = (e < o[np.newaxis, :]).sum(axis=0)
        equal = (e == o[np.newaxis, :]).sum(axis=0)
        rank  = below + (rng.random(o.shape[0]) * (equal + 1)).astype(int)
        counts += np.bincount(np.clip(rank, 0, K), minlength=K + 1)
    return counts

# Figure 1: reliability diagrams at the 90th and 99th percentile thresholds
fig_rel, axes_rel = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
for ax, (thr, thr_pct) in zip(axes_rel, [(Q90_MM, "90th"), (Q99_MM, "99th")]):
    ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Perfect calibration")
    for mk, label in [("cd", "CorrDiff"), ("cf", "CorrFlow")]:
        c, f, n = _reliability_curve(extreme_idxs["test"], "test", mk, thr)
        ok = np.isfinite(f) & (n > 0)
        ax.plot(c[ok], f[ok], "o-", color=colors[label], lw=2, ms=5, label=label)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Predicted exceedance probability")
    ax.set_ylabel("Observed exceedance frequency")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
rel_path = os.path.join(OUT_DIR, "reliability.pdf")
fig_rel.savefig(rel_path, dpi=300, bbox_inches="tight")
fig_rel.savefig(rel_path.replace(".pdf", ".png"), dpi=300, bbox_inches="tight"); plt.close(fig_rel)
print(f"OK  {rel_path}")

# Figure 2: verification rank histogram over wet land pixels
fig_rh, axr = plt.subplots(figsize=(7, 5), constrained_layout=True)
width = 0.4
for j, (mk, label) in enumerate([("cd", "CorrDiff"), ("cf", "CorrFlow")]):
    counts = _rank_hist(extreme_idxs["test"], "test", mk)
    freq = counts / max(counts.sum(), 1)
    axr.bar(np.arange(K_ENS + 1) + (j - 0.5) * width, freq, width=width,
            color=colors[label], alpha=0.85, label=label)
axr.axhline(1.0 / (K_ENS + 1), color="k", ls="--", lw=1, label="Flat (calibrated)")
axr.set_xlabel("Observation rank among ensemble members")
axr.set_ylabel("Frequency")
axr.legend(fontsize=8)
rh_path = os.path.join(OUT_DIR, "rank_histogram.pdf")
fig_rh.savefig(rh_path, dpi=300, bbox_inches="tight")
fig_rh.savefig(rh_path.replace(".pdf", ".png"), dpi=300, bbox_inches="tight"); plt.close(fig_rh)
print(f"OK  {rh_path}")


## §11b Per-timestep skill distributions

Post-processing of the cached extreme-subset predictions from `_get()` (the same
cache as Tables 2–3 and §11), so no inference or GPU is needed. For each extreme
test timestep it computes the ensemble-mean MAE, the CRPS, the mean ensemble
spread, and the ensemble-mean RMSE, and plots their distributions. This shows the
accuracy-dispersion trade-off that the means in Tables 2–3 average out.

In [ ]:
# Per-timestep skill distributions: box plots + spread-skill
# Reuses the cached extreme-subset predictions from _get() (same cache as Tables 2-3,
# reliability, rank histogram), so no inference or GPU is needed.
from matplotlib.lines import Line2D
C_CD, C_CF = "#FF9800", "#E53935"   # CorrDiff=orange, CorrFlow=red (matches PSD/rank figs)

def _per_timestep_stats(split, idxs):
    rows, n_skip = [], 0
    for zi in idxs:
        got = _get(split, zi)
        if got is None:
            n_skip += 1; continue
        truth, unet, cd, cf = got
        lv = (~np.isnan(truth)) & np.asarray(land_mask, bool)
        if not lv.any():                       # no valid land pixels this timestep
            n_skip += 1; continue
        rec = {"zi": int(zi)}
        with warnings.catch_warnings():        # ocean pixels are all-NaN across members: expected
            warnings.simplefilter("ignore", RuntimeWarning)
            for tag, ens in (("cd", cd), ("cf", cf)):
                m   = np.nanmean(ens, axis=0)  # ensemble mean (H,W)
                err = m - truth
                rec[f"mae_{tag}"]    = float(np.nanmean(np.abs(err[lv])))
                rec[f"crps_{tag}"]   = _crps_ens(truth, ens)
                rec[f"spread_{tag}"] = float(np.nanmean(np.nanstd(ens, axis=0)[lv]))
                rec[f"rmse_{tag}"]   = float(np.sqrt(np.nanmean(err[lv] ** 2)))
        rows.append(rec)
    return pd.DataFrame(rows), n_skip

_df_dist, _n_skip = _per_timestep_stats("test", extreme_idxs["test"])

# NaN sanity check: drop any timestep with a non-finite stat, report the count
_stat_cols = ["mae_cd","mae_cf","crps_cd","crps_cf","spread_cd","spread_cf","rmse_cd","rmse_cf"]
_n_before = len(_df_dist)
_df_dist = _df_dist.replace([np.inf, -np.inf], np.nan).dropna(subset=_stat_cols).reset_index(drop=True)
_n_nan = _n_before - len(_df_dist)
_df_dist.to_csv(os.path.join(OUT_DIR, "per_timestep_dist_test.csv"), index=False)
print(f"Extreme test timesteps requested : {len(extreme_idxs['test']):,}")
print(f"  skipped (no cache / no land)   : {_n_skip}")
print(f"  dropped (NaN/inf stat)         : {_n_nan}")
print(f"  used in figures                : {len(_df_dist):,}")
assert len(_df_dist) > 0, "no valid timesteps!"

# head-to-head win rate (per timestep)
w_mae  = float((_df_dist["mae_cf"]  < _df_dist["mae_cd"]).mean())
w_crps = float((_df_dist["crps_cf"] < _df_dist["crps_cd"]).mean())
print(f"\nCorrFlow beats CorrDiff on MAE  in {w_mae*100:4.1f}% of timesteps")
print(f"CorrFlow beats CorrDiff on CRPS in {w_crps*100:4.1f}% of timesteps")
for col in _stat_cols:
    v = _df_dist[col].values
    q1, q3 = np.percentile(v, 25), np.percentile(v, 75)
    fence = q3 + 1.5 * (q3 - q1)                      # upper whisker fence (1.5*IQR)
    whisker_top = v[v <= fence].max()                # actual whisker cap (largest point <= fence)
    print(f"  {col:9s} median={np.median(v):.4f}  Q1={q1:.4f}  Q3={q3:.4f}  "
          f"p90={np.percentile(v,90):.4f}  whisker_top={whisker_top:.4f}  max={v.max():.4f}")

# Figure A: MAE & CRPS box plots
fig, (axL, axR) = plt.subplots(1, 2, figsize=(10, 4.6), constrained_layout=True)
def _box(ax, cols, ylabel):
    data = [_df_dist[cols[0]].values, _df_dist[cols[1]].values]
    bp = ax.boxplot(data, patch_artist=True, widths=0.6, showfliers=True,
                    medianprops=dict(color="black", lw=1.6),
                    flierprops=dict(marker="o", ms=3, mfc="gray", mec="none", alpha=0.4))
    for patch, c in zip(bp["boxes"], (C_CD, C_CF)):
        patch.set_facecolor(c); patch.set_alpha(0.65); patch.set_edgecolor(c)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["CorrDiff", "CorrFlow"])
    ax.set_ylabel(ylabel); ax.grid(axis="y", alpha=0.3)
_box(axL, ["mae_cd", "mae_cf"],   "Per-timestep MAE (mm hr$^{-1}$)")
_box(axR, ["crps_cd", "crps_cf"], "Per-timestep CRPS (mm hr$^{-1}$)")
# explicit colour->model legend so the mapping is unambiguous
_leg = [Line2D([0],[0], marker="s", ls="none", ms=10, mfc=C_CD, mec=C_CD, label="CorrDiff"),
        Line2D([0],[0], marker="s", ls="none", ms=10, mfc=C_CF, mec=C_CF, label="CorrFlow")]
fig.legend(handles=_leg, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.06))
_bp_path = os.path.join(OUT_DIR, "boxplot_mae_crps.pdf")
plt.savefig(_bp_path, dpi=300, bbox_inches="tight")
plt.savefig(_bp_path.replace(".pdf", ".png"), dpi=300, bbox_inches="tight")
plt.show(); print("saved", _bp_path)

# Figure B: spread-skill
# Calibrated ensemble: spread ~ ensemble-mean RMSE (points on 1:1).
# Under-dispersion pushes points above 1:1 (RMSE larger than spread).
fig2, ax2 = plt.subplots(figsize=(5.6, 5.4), constrained_layout=True)
lim = 0.0
for tag, c, lab in (("cd", C_CD, "CorrDiff"), ("cf", C_CF, "CorrFlow")):
    x = _df_dist[f"spread_{tag}"].values; y = _df_dist[f"rmse_{tag}"].values
    ax2.scatter(x, y, s=10, c=c, alpha=0.35, edgecolors="none")
    lim = max(lim, float(np.nanpercentile(np.r_[x, y], 99)))
ax2.plot([0, lim], [0, lim], "k--", lw=1.2)
ax2.set_xlim(0, lim); ax2.set_ylim(0, lim)
ax2.set_xlabel("Ensemble spread (mm hr$^{-1}$)")
ax2.set_ylabel("Ensemble-mean RMSE (mm hr$^{-1}$)")
# opaque proxy handles so the colours are readable in the legend
_leg2 = [Line2D([0],[0], marker="o", ls="none", ms=8, mfc=C_CD, mec="none", label="CorrDiff"),
         Line2D([0],[0], marker="o", ls="none", ms=8, mfc=C_CF, mec="none", label="CorrFlow"),
         Line2D([0],[0], ls="--", c="k", lw=1.2, label="1:1 (calibrated)")]
ax2.legend(handles=_leg2, frameon=False, loc="lower right")
_ss_path = os.path.join(OUT_DIR, "spread_skill.pdf")
plt.savefig(_ss_path, dpi=300, bbox_inches="tight")
plt.savefig(_ss_path.replace(".pdf", ".png"), dpi=300, bbox_inches="tight")
plt.show(); print("saved", _ss_path)


## §12 Figure: single-step ODE trajectory (CorrFlow)

In [ ]:
# ODE trajectory figure
# Layout: equations at top, then two columns (circle above, map below),
# with a wide gap + big arrow between them.
# Requires §0 (setup and zarr) and §8 (map utilities).

import matplotlib.patches as mpatches

_prog_path = os.path.join(OUT_DIR, 'progressive_data.npz')
assert os.path.exists(_prog_path), f'progressive_data.npz not found at {_prog_path}'

_prog      = np.load(_prog_path)
mu_z_np    = _prog['mu_z_np']
CAPTURE_AT = list(_prog['capture_at'].astype(int))
_M_OUT     = float(_prog['M_OUT'])
_S_OUT     = float(_prog['S_OUT'])

frames_mm = {}
for si in CAPTURE_AT:
    mm = np.clip(
        np.expm1((mu_z_np + _prog[f'inter_{si}']) * (_S_OUT + 1e-6) + _M_OUT),
        0, None)
    frames_mm[si] = np.where(land_mask, mm, np.nan)

t0_mm = frames_mm[CAPTURE_AT[0]]
t1_mm = frames_mm[CAPTURE_AT[-1]]

ODE_EXTENT = [102.0, 116.5, 8.2, 23.8]

def _setup_ode_map(ax, proj):
    ax.set_facecolor(OCEAN_COLOR)
    ax.add_feature(_LAND)
    ax.add_feature(_COAST)
    ax.add_feature(_BORDS)
    ax.set_extent(ODE_EXTENT, crs=proj)
    # South China Sea label + Hoang Sa/Truong Sa markers, same placement logic as _setup_ax
    extent = ODE_EXTENT
    if extent[0] < 113 and extent[3] > 13:
        ax.text(113.2, 13.4, 'South China Sea', fontsize=9.5, color='#4fa3ff', alpha=0.4,
                ha='center', va='center', rotation=25, transform=proj, zorder=4)
    for la, lo, vn, en in _ISLES:
        if extent[0] < lo < extent[1] and extent[2] < la < extent[3]:
            if vn == 'Trường Sa':
                dot_la, dot_lo = la + 0.6, lo + 0.6
                ax.text(dot_lo - 0.2, dot_la - 0.2, f'{vn}\n{en}', transform=proj,
                        fontsize=7, color='#111', zorder=9, ha='right', va='top',
                        bbox=dict(fc='white', alpha=0.7, ec='#aaa', lw=0.3,
                                pad=1, boxstyle='round,pad=0.3'))
            else:
                ax.text(lo + 0.25, la + 0.25, f'{vn}\n{en}', transform=proj, fontsize=7,
                        color='#111', zorder=9,
                        bbox=dict(fc='white', alpha=0.7, ec='#aaa', lw=0.3,
                                pad=1, boxstyle='round,pad=0.3'))

C_DARK  = '#1A1A1A'
BG_ODE  = '#F8F6F0'   # circle fill
BG_FIG  = '#FFFFFF'   # white figure background
proj   = ccrs.PlateCarree()

FIG_W, FIG_H = 14.0, 9.0
fig_ode = plt.figure(figsize=(FIG_W, FIG_H), facecolor=BG_FIG)

# Layout
COL_W   = 0.25                          # width of each column (circle = map)
GAP_M   = 0.60                          # wide middle gap for the arrow
margin  = (1.0 - 2*COL_W - GAP_M) / 2
x_left  = margin                        # left column x
x_right = margin + COL_W + GAP_M       # right column x
mid_x   = 0.5

MAP_H   = 0.32                        # map height (figure fraction)
CIRC_H  = COL_W * FIG_W / FIG_H        # circle height, square in inches
GAP_ROW = 0.02                          # gap between circle and map

Y_MAP   = 0.05                          # bottom of maps
Y_CIRC  = Y_MAP + MAP_H + GAP_ROW      # bottom of circles
arrow_y = Y_CIRC + CIRC_H / 2          # arrow at circle midheight
TTL_Y   = arrow_y + 0.08
EQ_Y    = TTL_Y + 0.06

# Header text
fig_ode.text(mid_x, EQ_Y,
    r'CORRFLOW FULL PREDICTION TRAJECTORY',
    ha='center', va='bottom', fontsize=20, color='#111', fontweight='bold')

fig_ode.text(mid_x, TTL_Y,
    r'$\hat{x}(t) = \hat{\mu}(y) + r_t$'
    r'$\quad\vert\quad$'
    r'$d\mathbf{r} = v_\theta(\mathbf{r}_t,\;t,\;y)\,dt$',
    ha='center', va='bottom', fontsize=18, color=C_DARK)

# Big arrow between circles
fig_ode.add_artist(mpatches.FancyArrowPatch(
    (x_left  + COL_W + 0.02, arrow_y),
    (x_right - 0.02,          arrow_y),
    arrowstyle='-|>', mutation_scale=30, color=C_DARK, lw=2.0,
    transform=fig_ode.transFigure, zorder=5))

fig_ode.text(mid_x, arrow_y + 0.02,
    r'1 Euler step,  $\Delta t = 1.0$',
    ha='center', va='bottom', fontsize=18, color='#555')

# Circles
def _ode_circle(ax, l1, l2, l3):
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal'); ax.axis('off')
    ax.add_patch(plt.Circle((0, 0), 1.0, fc=BG_ODE, ec=C_DARK, lw=3.0, zorder=3))
    ax.text(0,  0.35, l1, ha='center', va='center',
            fontsize=26, fontweight='bold', color=C_DARK, zorder=4)
    ax.text(0, -0.15, l2, ha='center', va='center',
            fontsize=20, color='#333', zorder=4)
    ax.text(0, -0.65, l3, ha='center', va='center',
            fontsize=18, color='#666', zorder=4)

_ode_circle(fig_ode.add_axes([x_left,  Y_CIRC, COL_W, CIRC_H]),
    r'$\hat{x}(0)$', r'$\hat{\mu}+\varepsilon$',
    r'$\varepsilon\!\sim\!\mathcal{N}(0,I)$')
_ode_circle(fig_ode.add_axes([x_right, Y_CIRC, COL_W, CIRC_H]),
    r'$\hat{x}(1)$', r'$\hat{\mu}+\hat{r}$', r'Final $\hat{x}$')

# Map panels
for x_pos, mm_data, label, sublabel in [
    (x_left,  t0_mm, r'$\hat{x}(0)$', 'NOISY START'),
    (x_right, t1_mm, r'$\hat{x}(1)$', 'FINAL PREDICTION'),
]:
    ax = fig_ode.add_axes([x_pos, Y_MAP, COL_W, MAP_H], projection=proj)
    ax.set_facecolor(OCEAN_COLOR)
    ax.pcolormesh(lon2d_fine, lat2d_fine, mm_data,
                  transform=proj, shading='auto', cmap=RAIN_CMAP, norm=RAIN_NORM,
                  rasterized=True)
    _setup_ode_map(ax, proj)
    for sp in ax.spines.values():
        sp.set_edgecolor('#aaa'); sp.set_linewidth(0.8)
    # Label below each map
    fig_ode.text(x_pos + COL_W / 2, Y_MAP - 0.02, f'{sublabel}',
                 ha='center', va='top', fontsize=18, color=C_DARK)

_ode_path = os.path.join(OUT_DIR, 'ode_trajectory.pdf')
fig_ode.savefig(_ode_path, bbox_inches='tight', facecolor=BG_FIG, pad_inches=0.04)
fig_ode.savefig(_ode_path.replace('.pdf', '.png'),
                dpi=300, bbox_inches='tight', facecolor=BG_FIG, pad_inches=0.04)
plt.close(fig_ode)
print(f'✅ {_ode_path}')

## §13 Case-study figures and Table case_metrics

In [ ]:
# Case-study selection: survey held-out 2025 extremes by season x region
# Documents how the case events were chosen.
# Requires extreme_idxs (extreme-metadata cell) to have run. Safe to skip on reruns.
_idxs  = np.asarray(extreme_idxs["test"])
_times = pd.to_datetime(ds_zarr["time"].values[_idxs])
_rows = []
for _s in range(0, len(_idxs), 400):                       # chunked to bound memory
    _sl  = _idxs[_s:_s+400]
    _fld = np.expm1(ds_zarr["tp"].isel(time=_sl).values)   # (n,H,W) mm/hr, NaN over ocean
    _flat = _fld.reshape(_fld.shape[0], -1)
    _pk   = np.nanmax(_flat, axis=1)
    _am   = np.nanargmax(np.where(np.isnan(_flat), -1.0, _flat), axis=1)
    _pr, _pc = np.unravel_index(_am, _fld.shape[1:])
    for _k in range(_fld.shape[0]):
        _rows.append(dict(idx=int(_sl[_k]), time=_times[_s+_k], peak=float(_pk[_k]),
                          lat=float(lat1d_fine[_pr[_k]]), lon=float(lon1d_fine[_pc[_k]])))
_df = pd.DataFrame(_rows)

def _region(lat):
    if lat >= 20: return "North (Red R. delta / highlands)"
    if lat >= 18: return "North-central"
    if lat >= 14: return "Central (Annamite / C. coast)"
    return "South-central / South"
def _season(m):
    return ("Pre-monsoon (MAM)" if m in (3,4,5) else
            "SW monsoon (JJAS)" if m in (6,7,8,9) else
            "NE monsoon / late (OND)" if m in (10,11,12) else "Winter (JF)")

_df["month"]  = _df["time"].dt.month
_df["region"] = _df["lat"].map(_region)
_df["season"] = _df["month"].map(_season)

print("=== Top 15 most intense 2025 held-out extremes ===")
print(_df.sort_values("peak", ascending=False).head(15)
        [["time","peak","lat","lon","region","season"]].to_string(index=False))
print("\n=== Diversity menu: strongest event per (season x region) ===")
_best = (_df.sort_values("peak", ascending=False)
            .groupby(["season","region"], as_index=False).first()
            .sort_values("peak", ascending=False))
print(_best[["season","region","time","peak","lat","lon"]].to_string(index=False))
print("\nSelected cases -> CASE_META above (record event + Apr/Aug/Nov 2025).")


In [ ]:
# 2x3 case figure (means only) + per-case spread arrays cached for the
# dedicated spread-comparison figure in the next cell.
#   Row 0: ERA5 input | ERA5-Land truth | ERA5-Bilinear
#   Row 1: U-Net      | CorrDiff mean   | CorrFlow mean
# One rainfall colorbar, no mixed orientation. Panel titles carry the model
# name only; the peak value sits in a small corner annotation.
import warnings
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors

case_metric_rows = []
spread_rows = []
SPREAD_CACHE = {}   # cname -> dict(cd_std, cf_std, ext, peak_ll, zoom_spec, time_str, cd_SD_peak, cf_SD_peak)
for cname, (cts, split_cs, zoom_spec, peak_ll, fname) in CASE_META.items():
    if cts is None:
        print(f"{cname}: timestep not found, skipping."); continue
    la0, la1, lo0, lo1 = zoom_spec
    ext  = FULL_EXTENT
    proj = ccrs.PlateCarree()
    ts       = pd.Timestamp(cts)
    zi       = int(np.argmin(np.abs(_times_pd - ts)))
    time_str = ts.strftime("%Y-%m-%d %H:%M UTC")

    truth, unet, cd_ens, cf_ens = _get(split_cs, zi)
    era5_c_mm, era5_lat, era5_lon = load_era5_coarse_native(cts)
    era5_lon2d_c, era5_lat2d_c = np.meshgrid(era5_lon, era5_lat)

    lm = np.asarray(land_mask, bool)
    with warnings.catch_warnings():                      # ocean pixels are all-NaN: expected
        warnings.simplefilter("ignore", RuntimeWarning)
        cd_mean = np.nanmean(cd_ens, axis=0)
        cf_mean = np.nanmean(cf_ens, axis=0)
        cd_std  = np.where(lm, np.nanstd(cd_ens, axis=0), np.nan)
        cf_std  = np.where(lm, np.nanstd(cf_ens, axis=0), np.nan)
    tp_c_log = ds_zarr["tp_coarse"].isel(time=int(zi)).values
    bl = np.where(land_mask, np.expm1(tp_c_log), np.nan)

    # no suptitle; the LaTeX caption describes the figure
    CASE_LABEL = f"{cname.upper()}: {time_str}  |  peak={np.nanmax(truth):.1f} mm/hr"
    ERA5_LO2D, ERA5_LA2D, ERA5_DATA = era5_lon2d_c, era5_lat2d_c, era5_c_mm

    def ADD_BOX(ax):
        if peak_ll:
            ax.plot(peak_ll[1], peak_ll[0], "*", color="lime", ms=10,
                    mec="black", mew=0.5, transform=proj, zorder=11)
        ax.add_patch(mpatches.Rectangle(
            (lo0, la0), lo1-lo0, la1-la0, transform=proj, facecolor="none",
            edgecolor="red", linewidth=1.8, linestyle="-", zorder=10))

    fig = plt.figure(figsize=(16, 10), constrained_layout=True)
    gs = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 1])

    ax_era5  = fig.add_subplot(gs[0, 0], projection=proj)
    ax_truth = fig.add_subplot(gs[0, 1], projection=proj)
    ax_bl    = fig.add_subplot(gs[0, 2], projection=proj)
    ax_un    = fig.add_subplot(gs[1, 0], projection=proj)
    ax_cd    = fig.add_subplot(gs[1, 1], projection=proj)
    ax_cf    = fig.add_subplot(gs[1, 2], projection=proj)

    def _peak_tag(ax, val):
        ax.text(0.02, 0.03, f"peak {val:.1f}", transform=ax.transAxes,
                fontsize=11, va="bottom", ha="left", zorder=12,
                bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="none", alpha=0.85))

    # six rainfall panels on one shared rainfall scale
    im_r = ax_era5.pcolormesh(ERA5_LO2D, ERA5_LA2D, ERA5_DATA, transform=proj,
                              shading="auto", cmap=RAIN_CMAP, norm=RAIN_NORM, rasterized=True)
    _setup_ax(ax_era5, proj, ext)
    ax_era5.set_title("Coarse ERA5 input", fontsize=15)
    _peak_tag(ax_era5, float(np.nanmax(ERA5_DATA)))
    ADD_BOX(ax_era5)
    for ax, data, label in [
        (ax_truth, truth,   "ERA5-Land truth"),
        (ax_bl,    bl,      "ERA5-Bilinear"),
        (ax_un,    unet,    "U-Net"),
        (ax_cd,    cd_mean, "CorrDiff mean"),
        (ax_cf,    cf_mean, "CorrFlow mean"),
    ]:
        ax.pcolormesh(lon2d_fine, lat2d_fine, data, transform=proj, shading="auto",
                      cmap=RAIN_CMAP, norm=RAIN_NORM, rasterized=True)
        _setup_ax(ax, proj, ext)
        ax.set_title(label, fontsize=15)
        _peak_tag(ax, float(np.nanmax(data)))
        ADD_BOX(ax)
    cb_r = fig.colorbar(im_r, ax=[ax_era5, ax_truth, ax_bl, ax_un, ax_cd, ax_cf],
                        shrink=0.7, extend="both", pad=0.01, location="right")
    cb_r.set_label("rainfall (mm/hr)", fontsize=14); cb_r.ax.tick_params(labelsize=13)

    # peak-region mask (within ~1.5 deg of the ERA5-Land peak pixel)
    if peak_ll:
        _plat, _plon = peak_ll
        _pk = (np.abs(lat2d_fine - _plat) <= 1.5) & (np.abs(lon2d_fine - _plon) <= 1.5) & lm
    else:
        _pk = lm

    # near-peak spread computed here so it can be cached for the comparison figure
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        cd_sd_pk = float(np.nanmean(cd_std[_pk]))
        cf_sd_pk = float(np.nanmean(cf_std[_pk]))

    # spread fields (+ near-peak SD) plotted in the dedicated comparison figure (next cell)
    SPREAD_CACHE[cname] = dict(cd_std=cd_std, cf_std=cf_std, ext=ext,
                               peak_ll=peak_ll, zoom_spec=zoom_spec, time_str=time_str,
                               cd_SD_peak=cd_sd_pk, cf_SD_peak=cf_sd_pk)

    out_f = os.path.join(OUT_DIR, f"{fname}.pdf")
    fig.savefig(out_f, dpi=300, bbox_inches="tight")
    fig.savefig(out_f.replace('.pdf', '.png'), dpi=300, bbox_inches="tight")
    if cname == "case1":
        preview_f = out_f.replace('.pdf', '_preview.png')
        fig.savefig(preview_f, dpi=72, bbox_inches="tight")
        from IPython.display import Image, display
        display(Image(preview_f))
    plt.close(fig)
    print(f"  {fname}.pdf + .png   [{CASE_LABEL}]")

    # collect spread stats (domain-wide; near-peak already computed above)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        cd_sd_dom, cf_sd_dom = float(np.nanmean(cd_std)), float(np.nanmean(cf_std))
    spread_rows.append({
        "case": cname,
        "cd_SD_dom": cd_sd_dom, "cf_SD_dom": cf_sd_dom,
        "ratio_dom": cd_sd_dom / max(cf_sd_dom, 1e-9),
        "cd_SD_peak": cd_sd_pk, "cf_SD_peak": cf_sd_pk,
        "ratio_peak": cd_sd_pk / max(cf_sd_pk, 1e-9),
        "truth_peak": float(np.nanmax(truth)),
        "cd_peak":    float(np.nanmax(cd_mean)),
        "cf_peak":    float(np.nanmax(cf_mean)),
    })

    for tag, pm, ens in [
        ("ERA5-Bilinear",    bl,      bl[np.newaxis]),
        ("CorrDiff-UNet",    unet,    unet[np.newaxis]),
        ("CorrDiff (Heun)",  cd_mean, cd_ens),
        ("CorrFlow (Euler)", cf_mean, cf_ens),
    ]:
        case_metric_rows.append({
            "case": cname, "model": tag,
            "MAE":        float(np.nanmean(np.abs(pm - truth))),
            "CRPS":       _crps_ens(truth, ens),
            "peak_pred":  float(np.nanmax(pm)),
            "peak_truth": float(np.nanmax(truth)),
        })

df_cm = pd.DataFrame(case_metric_rows)
df_cm.to_csv(os.path.join(OUT_DIR, "tab_case_metrics.csv"), index=False)
df_sp = pd.DataFrame(spread_rows)
df_sp.to_csv(os.path.join(OUT_DIR, "tab_case_spread.csv"), index=False)

print("\n=== Table case_metrics (MAE / CRPS / peaks) ===")
print(df_cm.round(3).to_string(index=False))

print("\n=== Ensemble spread: domain-wide vs near peak (mm/hr) ===")
print(df_sp.round(4).to_string(index=False))

print("\n" + "="*70)
print("  CASE STUDY FIGURES — peak values for LaTeX tables")
print("="*70)
for case in df_cm["case"].unique():
    sub = df_cm[df_cm["case"] == case]
    tp  = sub["peak_truth"].iloc[0]
    print(f"\n  {case.upper()}  truth peak={tp:.1f} mm/hr")
    for _, row in sub.iterrows():
        print(f"    {row['model']:<24} & {row['peak_pred']:.1f} & {tp:.1f} \\\\")
print("="*70)

In [ ]:
# Spread comparison: 3 cases (rows) x 2 models (columns), one shared colorbar
# on the right.
#
# Annotation shows the near-peak ensemble spread (cd_SD_peak / cf_SD_peak from
# the previous cell: mean SD within +/-1.5 deg of the peak pixel, land only),
# the same value quoted in the LaTeX cross-case text -- not the domain-mean.
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

# Vector text/lines in the PDF; the pcolormesh fields are rasterized at high dpi.
plt.rcParams["pdf.fonttype"] = 42

_cases = [c for c in ["case1", "case2", "case3"] if c in SPREAD_CACHE]
_row_label = {"case1": "Case 1\n28 Apr 2025", "case2": "Case 2\n25 Aug 2025",
              "case3": "Case 3\n6 Nov 2025"}

proj = ccrs.PlateCarree()
fig, axes = plt.subplots(len(_cases), 2, figsize=(12, 5 * len(_cases)),
                         subplot_kw={"projection": proj}, constrained_layout=True)
axes = np.atleast_2d(axes)

# one shared spread scale across all cases and both models, set by the CorrDiff spreads
_vmax = max(float(np.nanpercentile(SPREAD_CACHE[c]["cd_std"], 99)) for c in _cases)
norm_sd = mcolors.Normalize(vmin=0.0, vmax=max(_vmax, 1e-4))

_last_row = len(_cases) - 1
for r, cname in enumerate(_cases):
    d = SPREAD_CACHE[cname]
    la0, la1, lo0, lo1 = d["zoom_spec"]
    for c, (sd, mlabel) in enumerate([(d["cd_std"], "CorrDiff"), (d["cf_std"], "CorrFlow")]):
        ax = axes[r, c]
        im_s = ax.pcolormesh(lon2d_fine, lat2d_fine, sd, transform=proj, shading="auto",
                             cmap="viridis", norm=norm_sd, rasterized=True)
        gl = _setup_ax(ax, proj, d["ext"])
        # latitude labels only on the left column, longitude only on the bottom row
        if gl is not None:
            gl.left_labels   = (c == 0)
            gl.bottom_labels = (r == _last_row)
        if d["peak_ll"]:
            ax.plot(d["peak_ll"][1], d["peak_ll"][0], "*", color="lime", ms=10,
                    mec="black", mew=0.5, transform=proj, zorder=11)
        ax.add_patch(mpatches.Rectangle((lo0, la0), lo1 - lo0, la1 - la0, transform=proj,
                                        facecolor="none", edgecolor="red", linewidth=1.6, zorder=10))
        if r == 0:
            ax.set_title(mlabel, fontsize=16)
        _pk_sd = d["cd_SD_peak"] if mlabel == "CorrDiff" else d["cf_SD_peak"]
        ax.text(0.02, 0.03, f"peak SD {_pk_sd:.3f}", transform=ax.transAxes, fontsize=10.5,
                va="bottom", ha="left", zorder=12,
                bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="none", alpha=0.85))

    # Row label placed in offset points from the axes' left edge, so it clears the
    # latitude labels regardless of figsize (an axes-fraction offset would drift).
    axes[r, 0].annotate(_row_label[cname], xy=(0.0, 0.5), xycoords="axes fraction",
                        xytext=(-58, 0), textcoords="offset points",
                        fontsize=13, va="center", ha="center", rotation=90)

cb = fig.colorbar(im_s, ax=axes.ravel().tolist(), shrink=0.5, extend="max",
                  pad=0.02, location="right")
cb.set_label("ensemble spread (mm/hr)", fontsize=14); cb.ax.tick_params(labelsize=12)

out_f = os.path.join(OUT_DIR, "case_spread_comparison.pdf")
fig.savefig(out_f, dpi=300, bbox_inches="tight")          # rasterized maps at 300 dpi
fig.savefig(out_f.replace(".pdf", ".png"), dpi=300, bbox_inches="tight")
plt.close(fig)
print("  case_spread_comparison.pdf + .png")

## §14 Summary

In [ ]:
print("\n" + "="*70)
print("  ALL RESULTS — copy into LaTeX")
print("="*70)

print("\n--- Table 1: Full-period ---")
if "df_full" in dir() and df_full is not None:
    for spl in ["train","val","test"]:
        print(f"  {spl}:")
        for _,r in df_full[df_full.split==spl].iterrows():
            print(f"    {r.model:<22}  MAE={r.MAE:.3f}  CRPS={r.CRPS:.3f}")
else:
    print("  (cache_sample.py has not been run; skipping Table 1)")

print("\n--- Table 2: Extreme test (2025) ---")
for k in ("bl","un","cd","cf"):
    d=m_test[k]
    print(f"  {label_map[k]:<22}  MAE={d['mae']:.3f}  CRPS={d['crps']:.3f}  FSS={d['fss']:.3f}  BS={d['bs']:.3f}")

print("\n--- Table 3: Extreme train (2017-2023) ---")
for k in ("bl","un","cd","cf"):
    d=m_train[k]
    print(f"  {label_map[k]:<22}  MAE={d['mae']:.3f}  CRPS={d['crps']:.3f}  FSS={d['fss']:.3f}  BS={d['bs']:.3f}")

print("\n--- Table 6: CorrFlow ablation ---")
print(df_ab[["steps","MAE","CRPS","Latency_s"]].round(3).to_string(index=False))

print("\n--- Table case_metrics ---")
print(df_cm.round(3).to_string(index=False))

print("\n--- Figures ---")
for f in sorted(os.listdir(OUT_DIR)):
    if f.endswith(".pdf"): print(f"  {f}")
print(f"\nOUT_DIR: {OUT_DIR}")
